# Actividad 2B · Aprendizaje supervisado y no supervisado

**Tiempo estimado:** 20 minutos  
**Recorrido:** SQL + Python

En el notebook anterior distinguimos clasificación, regresión y agrupamiento. Ahora vamos a observar otra diferencia: **¿contamos con la respuesta que queremos predecir o buscamos descubrir grupos sin esa respuesta?**

Usaremos nuevamente el dataset de *churn* para comparar ambos enfoques, sin entrenar modelos todavía.

**Para empezar:** ejecutá la celda de preparación sin modificarla. El notebook descargará los datos y dejará lista la conexión con DuckDB.


## ¿Qué cambia entre ambos enfoques?

| Característica | Supervisado | No supervisado |
|---|---|---|
| ¿Hay una variable objetivo? | Sí | No |
| ¿Conocemos la respuesta de los casos de entrenamiento? | Sí | No |
| Problemas frecuentes | Clasificación y regresión | Agrupamiento |
| ¿Qué buscamos? | Predecir para casos nuevos | Identificar perfiles o estructuras en los datos |

En un problema **supervisado** distinguimos las variables predictoras (`X`) de la variable objetivo (`y`). En uno **no supervisado** trabajamos con las características de los casos, sin indicar una respuesta que el modelo deba aprender a predecir.


In [ ]:
#@title Preparación automática del entorno (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

print("Preparando el entorno...")

for package in ["duckdb", "pandas"]:
    if importlib.util.find_spec(package) is None:
        print(f"Instalando {package}...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import duckdb
import pandas as pd

print("Paquetes preparados correctamente.")

DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
DATA_PATH = Path("Telco-Customer-Churn.csv")

if not DATA_PATH.exists():
    print("Descargando dataset...")
    urlretrieve(DATA_URL, DATA_PATH)

churn = pd.read_csv(DATA_PATH)

required_columns = {"Churn", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "InternetService"}
if not required_columns.issubset(churn.columns):
    raise ValueError("El dataset descargado no contiene las columnas esperadas.")

churn["TotalCharges"] = pd.to_numeric(churn["TotalCharges"], errors="coerce")

# Si la celda se ejecuta otra vez, cerramos primero una conexión previa.
if "con" in globals():
    try:
        con.close()
    except Exception:
        pass

con = duckdb.connect(":memory:")
con.register("churn_df", churn)
con.execute("CREATE OR REPLACE TABLE churn AS SELECT * FROM churn_df")

clientes = f"{len(churn):,}".replace(",", ".")
print(f"Entorno listo: {clientes} clientes y {churn.shape[1]} variables.")


## 1. Aprendizaje supervisado · predictores y respuesta

Recuperamos algunas características de los clientes junto con `Churn`, que indica si abandonaron el servicio.

El código separa las **predictoras (`X`)** de la **variable objetivo (`y`)**. Observá ambas tablas: tenemos información de los clientes y también la respuesta registrada para esos casos.


In [ ]:
datos_supervisados = con.sql("""
SELECT tenure, MonthlyCharges, Contract, InternetService, Churn
FROM churn
LIMIT 8
""").df()

X = datos_supervisados.drop(columns=["Churn"])
y = datos_supervisados["Churn"]

print("Predictores X:")
display(X)
print("Variable objetivo y:")
display(y.to_frame())


## 2. Aprendizaje no supervisado · características sin respuesta

Ahora consultamos las mismas características, pero sin incluir `Churn` como variable objetivo.

Pensá qué perfiles podrían aparecer al comparar antigüedad, cargo mensual y tipo de contrato. **La consulta no realiza un agrupamiento**: solamente muestra los datos que utilizaríamos para buscar semejanzas.


In [ ]:
X_sin_objetivo = con.sql("""
SELECT tenure, MonthlyCharges, Contract, InternetService
FROM churn
LIMIT 8
""").df()

print("Datos disponibles sin variable objetivo:")
X_sin_objetivo


## Para pensar a partir de los datos

Respondé brevemente:

1. Si queremos predecir `Churn` para clientes nuevos, ¿qué enfoque corresponde? ¿Qué función cumple `Churn` en los datos de entrenamiento?
2. Si buscamos perfiles de clientes sin indicar previamente quién pertenece a cada grupo, ¿qué enfoque usaríamos?
3. ¿Por qué no podemos calcular el «porcentaje de aciertos» de un agrupamiento del mismo modo que en una clasificación con respuestas conocidas?
4. Si una variable contiene información que se registra **después** de la baja del cliente, ¿sería correcto usarla para anticipar `Churn`? Justificá tu respuesta.

**Tus respuestas:** escribilas acá. No necesitás entrenar modelos ni calcular métricas para resolver estas preguntas.


## Cierre

Antes de elegir una técnica, tenemos que precisar la pregunta y revisar qué información conocemos. En la Práctica de esta semana vamos a recorrer estas decisiones al construir un primer árbol de clasificación con otro dataset.


In [ ]:
# Cierre de la conexión con DuckDB
if "con" in globals():
    try:
        con.close()
    except Exception:
        pass
